In [ ]:
from anwm.projection import create_pose_matrix, euler_to_quaternion

In [ ]:
import os
import csv
import pickle
import numpy as np
import shutil
from pathlib import Path
import json

In [ ]:
root_dir = Path("/path/to/airvln")  # Set the local AirVLN root.
dataset_dir = root_dir / 'traj_obs'
status_dir = root_dir / 'waypoints'       # CSV 所在目录
depth_dir = root_dir / 'traj_obs'
output_suffix = '_processed'              # 输出轨迹文件夹的后缀名

In [ ]:
folders = [f for f in dataset_dir.iterdir() if f.is_dir()]
print("发现的轨迹文件夹：")
for f in folders:
    print(f)

In [ ]:
print(f'共{len(folders)}条轨迹')

In [ ]:
with open(status_dir / 'train.json', 'r', encoding='utf-8') as f:
    data = json.load(f)['episodes']
with open(status_dir / 'val_seen.json', 'r', encoding='utf-8') as f:
    val_data = json.load(f)['episodes']
data = data + val_data

In [ ]:
data[0]

In [ ]:
data[0].keys()

In [ ]:
for item in data:
    if item['scene_id'] == 16:
        trajectory_id = item['trajectory_id']
        reference_path = item['reference_path']

        # 设置输出路径
        csv_path = os.path.join(status_dir, f'{trajectory_id}.csv')

        # 写入 CSV
        with open(csv_path, 'w', newline='', encoding='utf-8') as csvfile:
            writer = csv.writer(csvfile)
            writer.writerow(['x', 'y', 'z', 'roll', 'pitch', 'yaw'])
            for point in reference_path:
                writer.writerow(point)

print(f"共导出 {len(data)} 个 trajectory 到文件夹：{status_dir}")

In [ ]:
csv_files = list(status_dir.glob('*.csv'))
print("发现的轨迹 CSV 文件：")
for f in csv_files:
    print(f.name)

In [ ]:
print(f'共{len(csv_files)}条轨迹')

In [ ]:
def compute_intrinsic_matrix(width, height, fov_x_degree):
    fov_x = np.radians(fov_x_degree)
    fov_y = 2 * np.arctan((height*1.0 / width) * np.tan(fov_x / 2))

    intrinsic_parameters = {
        'width': width,
        'height': height,
        'fx': width / (2 * np.tan(fov_x / 2)), # 1.5 * width,
        'fy': height / (2 * np.tan(fov_y / 2)), # 1.5 * width,
        'cx': width / 2,
        'cy': height / 2,
    }

    fx = intrinsic_parameters['fx']
    fy = intrinsic_parameters['fy']
    cx = intrinsic_parameters['cx']
    cy = intrinsic_parameters['cy']

    K = np.array([
        [fx,  0, cx],
        [ 0, fy, cy],
        [ 0,  0,  1]
    ])
    return K


fov_x_degree = 90
width = 512
height = 512
K = compute_intrinsic_matrix(width, height, fov_x_degree)
K

In [ ]:
t_ec = np.array(
    [
        [0.0, 0, 1, 0],
        [1.0, 0, 0, 0],
        [0.0, 1, 0, 0],
        [0.0, 0, 0, 1]
    ]
)

In [ ]:
all_dists_2d = []
all_dists = []

In [ ]:
for csv_file in csv_files:
    traj_name = csv_file.stem
    traj_path = dataset_dir / traj_name / 'rgb'
    depth_path = depth_dir / traj_name / 'dep'
    print(f"===================================================================={traj_name}==============================================================================================")
    output_dir = root_dir / 'outputs' / f"{traj_name}{output_suffix}"

    positions = []
    points = []
    orientations = []
    pitch_list = []
    roll_list = []
    timestamps = []
    images = []
    depth_list = []
    pose_list = []
    image_copy_tasks = []
    frame_counter = 0
    with open(csv_file, 'r') as f:
        reader = csv.DictReader(f)
        for idx, row in enumerate(reader):
            timestamp = idx
            if timestamp > 10:
                x, y, z = float(row['x']), float(row['y']), float(row['z'])
                pitch = float(row['pitch'])
                roll = float(row['roll'])
                yaw = float(row['yaw'])  # 只取 yaw（朝向）
                image_src = traj_path / f'rgb_obs_front_{timestamp}.png'
                # image_dst = output_dir / f"{idx}.jpg"
                image_dst = output_dir / f"{frame_counter}.jpg"
                depth_src = depth_path / f"dep_obs_front_{timestamp}.npy"
        
                if not image_src.exists():
                    # print(f"❌ 缺失图像：{image_src}")
                    break
                else:
                    image_copy_tasks.append((image_src, image_dst))
                    
                depth = np.load(depth_src)
                depth = np.squeeze(depth, axis=-1)
                positions.append(np.array([x, y]))
                points.append(np.array([x, y, z]))
                orientations.append(yaw) 
                pitch_list.append(pitch)
                roll_list.append(roll)
                timestamps.append(float(timestamp))
                images.append(f"{frame_counter}")
                depth_list.append(depth)
                translation = np.array([x, y, z])
                quaternion = euler_to_quaternion(yaw, pitch, roll)
                pose_src = create_pose_matrix(translation, quaternion)
                pose_src = pose_src.dot(t_ec)
                pose_list.append(pose_src)

                # print(f"{timestamp} - {idx} - {frame_counter}.jpg")
                frame_counter += 1
                
    valid_csv_rows = timestamp - 10  # + 1 - 11
    if frame_counter == 0:
        print("无可用帧！")
        continue
    if valid_csv_rows != frame_counter:
        print(f"📊 {traj_name} 有效CSV行数: {valid_csv_rows}")
        print(f"🖼️ {traj_name} 复制图像数: {frame_counter}")
        print(f"⚠️ 不一致：{traj_name} 的有效行数 ≠ 图像数！")
        continue
    # 复制帧
    output_dir.mkdir(parents=True, exist_ok=True)
    for src, dst in image_copy_tasks:
        shutil.copy(src, dst)

    # 多维通用 - 二维
    dists_2d = [np.linalg.norm(positions[i] - positions[i - 1]) for i in range(1, len(positions))]
    all_dists_2d.extend(dists_2d)
    # 三维
    dists = [np.linalg.norm(points[i] - points[i - 1]) for i in range(1, len(points))]
    all_dists.extend(dists)

    # 写入 traj_data.pkl
    traj_data = {
        'K': np.array(K),
        'position': np.array(positions),    # 位置：[T, 2]
        'yaw': np.array(orientations),      # 航向：[T,]
        'timestamps': np.array(timestamps),
        'images': np.array(images),                   # 图像列表
        'pitch': np.array(pitch_list),
        'roll': np.array(roll_list),
        'point': np.array(points),          # 位置：[T, 3]
        'depth': np.array(depth_list),
        'pose': np.array(pose_list),
    }

    with open(output_dir / 'traj_data.pkl', 'wb') as f:
        pickle.dump(traj_data, f)

    print(f"✅ 转换完成：{output_dir}")

In [ ]:
folders = [f for f in (root_dir / 'outputs').iterdir() if f.is_dir()]
print(f'处理完成共{len(folders)}条轨迹')
print("发现的轨迹文件夹：")
for f in folders:
    print(f)

In [ ]:
waypoint_spacing = float(np.mean(all_dists_2d))
print(f"Average spacing between waypoints (meters) in 2d plane: {waypoint_spacing} m.")
waypoint_spacing = float(np.mean(all_dists))
print(f"Average spacing between waypoints (meters): {waypoint_spacing} m.")

In [ ]:
check_dir = folders[0]
print("样例输出文件夹：", check_dir)

with open(check_dir / 'traj_data.pkl', 'rb') as f:
    sample_data = pickle.load(f)

# 输出检查内容
print("数据结构：", sample_data.keys())
print("内参示例：", sample_data['K'])
print("位置示例：", sample_data['position'][:2])
print("航向角示例：", sample_data['yaw'][:2])
print("时间戳示例：", sample_data['timestamps'][:2])
print("图像索引示例：", sample_data['images'][:2])
print("pitch 示例：", sample_data['pitch'][:2])
print("roll 示例：", sample_data['roll'][:2])
print("三维位置示例：", sample_data['point'][:2])

In [ ]:
print("深度数据示例：", sample_data['depth'][:2])
print("深度数据形状:", sample_data['depth'][:2].shape)

In [ ]:
print("姿态示例：", sample_data['pose'][:2])